# LegalTrustRAG: fine-tune the legal reranker on LegalBench-RAG (Colab, T4)

Closes research gap 3 (synopsis section 6.3): fine-tunes `BAAI/bge-reranker-base` on LegalBench-RAG
question/clause pairs, evaluates it against the untuned model on held-out documents, and exports it
to the ONNX layout the Next.js app loads with `@huggingface/transformers`.

**Before you start:** `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`.

Rough timings on a T4 (estimates, not measured on Colab): install 3 min, data 1 min, training on
`mini` 2 epochs ~5 min (on `full` ~12 min per epoch), export 3 min, evaluation 2-8 min.
See `training/README.md` for details.

## 1. Settings
`VARIANT = "mini"` trains on LegalBench-RAG-mini (776 queries, quick). `"full"` uses all ~6.9k queries.
`"full-valmini"` trains on the full benchmark minus the mini documents and validates on exactly the
LegalBench-RAG-mini queries (corpus-wide BM25 candidates): the configuration to quote in the report.

If the repository is private, add a Colab secret named `GITHUB_TOKEN` (key icon in the left bar) with
read access to it. The `training/` folder must be pushed to `BRANCH` first.

In [ ]:
REPO_URL = "https://github.com/ItisShreyadav/legaltrustrag.git"
BRANCH = "main"
VARIANT = "mini"            # "mini" | "full" | "full-valmini"
EPOCHS = 2                  # 2 for mini; use 1 for full / full-valmini to stay near one hour
INCLUDE_FP32_IN_ZIP = False # the app uses the int8 model (onnx/model_quantized.onnx) by default

PREP_FLAGS = {"mini": "", "full": "--full", "full-valmini": "--full --val-mini --candidate-scope corpus"}[VARIANT]
DATA_DIR = f"training/data/{VARIANT}"
print(VARIANT, PREP_FLAGS, DATA_DIR, EPOCHS)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: Runtime -> Change runtime type -> T4 GPU"

## 2. Get the code

In [ ]:
import os, subprocess
os.chdir("/content")
url = REPO_URL
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if token:
        url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
except Exception:
    pass
if not os.path.exists("/content/legaltrustrag"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, "/content/legaltrustrag"], check=True)
os.chdir("/content/legaltrustrag")
assert os.path.exists("training/prepare_data.py"), "training/ not found on this branch: push it first or upload it"
!git log -1 --oneline

## 3. Install the Python requirements (keeps Colab's CUDA build of torch)

In [ ]:
!pip install -q -r training/requirements.txt
!python -c "import torch, transformers, sentence_transformers as st; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), '| transformers', transformers.__version__, '| sentence-transformers', st.__version__)"

## 4. Build the training data (downloads LegalBench-RAG, ~90 MB)

In [ ]:
!python training/prepare_data.py {PREP_FLAGS}

## 5. Fine-tune (evaluates the untuned model first, keeps the best epoch by MRR@10)

In [ ]:
!python training/train_reranker.py --data {DATA_DIR} --epochs {EPOCHS}

## 6. Export to the transformers.js layout (`onnx/model.onnx` + int8 `onnx/model_quantized.onnx`)

In [ ]:
!python training/export_onnx.py --model training/out/legal-reranker --out public/models/legal-reranker

## 7. Held-out evaluation for the report
BM25 vs. untuned `bge-reranker-base` vs. the fine-tuned model on all held-out queries (GPU, PyTorch),
written to `training/out/eval_results.{json,md}`. The second cell checks that the exported ONNX graphs the
app runs (fp32 and int8) rank like the PyTorch model; ONNX runs on the CPU here, so it uses a 20-query,
top-10-candidate subset (a few minutes) and an fp32 PyTorch row on the same subset for comparison.

In [ ]:
!python training/eval_reranker.py --data {DATA_DIR} --model BAAI/bge-reranker-base --model training/out/legal-reranker --out training/out/eval_results.json

In [ ]:
!python training/eval_reranker.py --data {DATA_DIR} --model training/out/legal-reranker --onnx public/models/legal-reranker --limit 20 --top-candidates 10 --no-fp16 --out training/out/onnx_parity.json

## 8. Download the model (unzip into `public/models/` of the app) and the evaluation files

In [ ]:
import shutil, zipfile, os
os.makedirs("/content/artifacts", exist_ok=True)
model_zip = "/content/artifacts/legal-reranker.zip"
with zipfile.ZipFile(model_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk("public/models/legal-reranker"):
        for f in files:
            p = os.path.join(root, f)
            if f == "model.onnx" and not INCLUDE_FP32_IN_ZIP:
                continue
            z.write(p, os.path.relpath(p, "public/models"))
eval_zip = "/content/artifacts/legal-reranker-eval.zip"
with zipfile.ZipFile(eval_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for p in ["training/out/eval_results.json", "training/out/eval_results.md",
              "training/out/onnx_parity.json", "training/out/onnx_parity.md",
              "training/out/legal-reranker/eval_results.json", "training/out/legal-reranker/eval_results.md",
              f"{DATA_DIR}/stats.json", "public/models/legal-reranker/export_info.json"]:
        if os.path.exists(p):
            z.write(p, p.replace("/", "__"))
for p in (model_zip, eval_zip):
    print(p, f"{os.path.getsize(p) / 1e6:.0f} MB")
print(open("training/out/eval_results.md").read())

In [ ]:
from google.colab import files
files.download("/content/artifacts/legal-reranker-eval.zip")
files.download("/content/artifacts/legal-reranker.zip")

Optional: copy the artifacts to Google Drive instead of downloading (large files download more reliably this way).

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# !mkdir -p /content/drive/MyDrive/legaltrustrag && cp /content/artifacts/*.zip /content/drive/MyDrive/legaltrustrag/

## 9. Use it in the app
```bash
unzip legal-reranker.zip -d public/models/          # -> public/models/legal-reranker/{config.json,tokenizer.json,onnx/...}
LTR_RERANK_MODEL=public/models/legal-reranker npm run dev
# or: LTR_LOCAL_MODELS_DIR=public/models LTR_RERANK_MODEL=legal-reranker npm run dev
```
The server logs `[retrieval] reranker loaded: public/models/legal-reranker (local ..., dtype q8)`.